# Latent Diffusion Model (LDM)

Stable Diffusion 与 DDPM 的区别在于，Stable Diffusion 是在潜空间中进行 diffusion，得到去噪后的 latent features 之后，再利用 VAE 对其进行解码，得到真实的图像。

In [11]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import CLIPTokenizer, CLIPTextModel
from datasets import load_dataset

from ldm import StableDiffusion

In [12]:
# hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = 512
latent_size = 64
in_channels = 3
epochs = 1000
batch_size = 16
lr = 1e-4
T = 1000
save_checkpoint = 100
lambda_cons = 0.1   # consistency loss weight
max_lambda_cons = 1.0   # max consistency loss weight

In [13]:
# load clip
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

In [14]:
# load model
model = StableDiffusion(in_channels=in_channels, latent_dim=4, image_size=image_size, timesteps=T, device=device)
model.load_vae("../logs/vae/vae.pth")
model.freeze_vae()
model.to(device)

StableDiffusion(
  (vae): VAE(
    (encoder): Sequential(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (1): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
      (2): Sequential(
        (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2)
      )
    )
    (mu): Conv2d(256, 4, kernel_size=(1, 1), stride=(1, 1))
    (log_var): Conv2d(256, 4, kernel_size=(1, 1), stride=(1, 1))
    (decoder): Sequential(
      (0): ConvTranspose2d(4, 256, kernel_

In [15]:
def transform_all(data):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    images = [transform(image.convert('RGB')) for image in data["image"]]
    en_texts = data["en_text"]
    return {"image": images, "en_text": en_texts}

In [16]:
# dataset
class LatentDataset(Dataset):
    def __init__(self, original_dataset, vae, device):
        self.original_dataset = original_dataset
        self.vae = vae
        self.device = device
        self.transforms = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)
        ])

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, index):
        data = self.original_dataset[index]
        image = data["image"]
        en_text = data["en_text"]
        image = self.transforms(image)
        image = image.to(self.device)
        with torch.no_grad():
            latent = self.vae.encode(image.unsqueeze(0))[0]
        return {"image": image, "en_text": en_text, "latent": latent.squeeze(0).cpu()}

In [17]:
# dataset
dataset = load_dataset("svjack/pokemon-blip-captions-en-zh", split="train", cache_dir=r"D:\HuggingFace\cache")
dataset.set_transform(transform_all)
train_dataset = LatentDataset(dataset.select(range(600)), model.vae, device)
val_dataset = dataset.select(range(600, 800))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=True)

In [18]:
optimizer = Adam(model.parameters(), lr=lr, weight_decay=5e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

save_path = "../logs/ldm"
os.makedirs(save_path, exist_ok=True)

In [19]:
def diversity_loss(latents, use_cos=False):
    batch_size = latents.shape[0]
    latents = latents.view(batch_size, -1)

    if use_cos:
        latents = F.normalize(latents, p=2, dim=1)
        similarity = torch.matmul(latents, latents.T)
    else:
        similarity = torch.matmul(latents, latents)
    similarity = similarity - torch.eye(batch_size, device=latents.device)

    return similarity.sum() / (batch_size * (batch_size - 1))

In [20]:
diversity_weight = 0.01

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")

    model.train()
    train_loss = 0.0
    num_batches = 0
    current_lambda_cons = min(lambda_cons * (epoch + 1) / epochs, max_lambda_cons)
    for batch in tqdm(train_loader, desc="Training"):
        latents = batch['latent'].to(device)
        text = batch['en_text']

        timesteps = torch.randint(0, T, (latents.shape[0],), device=device).long()
        noisy_latent, noise = model.noise_scheduler.add_noise(latents, timesteps)

        text_inputs = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
        text_embeddings = text_encoder(text_inputs["input_ids"].to(device)).last_hidden_state

        # compute loss
        noise_pred = model(noisy_latent, timesteps, text_embeddings)
        mse_loss = F.mse_loss(noise_pred, noise)
        div_loss = diversity_loss(noisy_latent, use_cos=True)

        alpha_t = model.noise_scheduler.alpha[timesteps][:, None, None, None]
        sqrt_alpha_t = torch.sqrt(alpha_t)
        sqrt_1_minus_alpha_t = torch.sqrt(1. - alpha_t)
        latents_pred = (noisy_latent - sqrt_1_minus_alpha_t * noise_pred) / sqrt_alpha_t
        cons_loss = F.mse_loss(latents_pred, latents)

        total_loss = mse_loss + diversity_weight * div_loss + current_lambda_cons * cons_loss
        train_loss += total_loss.item()
        num_batches += 1

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            diversity_weight = min(diversity_weight * 1.05, 0.1)
    train_loss /= num_batches

    model.eval()
    val_loss = 0.0
    num_batches = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            images = batch['image'].to(device)
            latents = model.encode(images)
            text = batch['en_text']

            timesteps = torch.randint(0, T, (latents.shape[0],), device=device).long()
            noisy_latent, noise = model.noise_scheduler.add_noise(latents, timesteps)

            text_inputs = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
            text_embeddings = text_encoder(text_inputs["input_ids"].to(device)).last_hidden_state

            # compute loss
            noise_pred = model(noisy_latent, timesteps, text_embeddings)
            mse_loss = F.mse_loss(noise_pred, noise)

            val_loss += mse_loss.item()
            num_batches += 1
    val_loss /= num_batches
    scheduler.step(train_loss)

    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if (epoch + 1) % save_checkpoint == 0:
        torch.save(model.state_dict(), os.path.join(save_path, f"ldm_{epoch+1}.pth"))
        model.eval()
        with torch.no_grad():
            sample_text = ["a cartoon pikachu with big eyes and big ears"]
            text_inputs = tokenizer(sample_text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
            text_embeddings = text_encoder(text_inputs["input_ids"].to(device)).last_hidden_state
            sampled_latent = model.sample(text_embeddings, latent_size, len(sample_text), guidance_scale=7.5)
            sampled_image = model.decode(sampled_latent)

            for i, image in enumerate(sampled_image):
                image = torch.clip((image + 1) / 2, 0, 1)
                image = image.cpu().permute(1, 2, 0).numpy()
                image = (image * 255).astype(np.uint8)
                image_pil = Image.fromarray(image)
                image_pil.save(os.path.join(save_path, f"image_{epoch+1}_sample_{i}.png"))

torch.save(model.state_dict(), os.path.join(save_path, f"ldm_latest.pth"))

Epoch 1/1000


Validation: 100%|██████████| 12/12 [00:04<00:00,  2.76it/s]


Train Loss: 0.4072, Val Loss: 0.5572
Epoch 2/1000


Validation: 100%|██████████| 12/12 [00:03<00:00,  3.16it/s]


Train Loss: 0.3479, Val Loss: 0.4530
Epoch 3/1000


...

Train Loss: 0.1786, Val Loss: 0.2065
Epoch 999/1000


Validation: 100%|██████████| 12/12 [00:04<00:00,  2.56it/s]


Train Loss: 0.1698, Val Loss: 0.2430
Epoch 1000/1000


Validation: 100%|██████████| 12/12 [00:04<00:00,  2.76it/s]


Train Loss: 0.1715, Val Loss: 0.2302


In [23]:
model.eval()
with torch.no_grad():
    sample_text = ["a drawing of a green pokemon with red eyes"]
    text_inputs = tokenizer(sample_text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
    text_embeddings = text_encoder(text_inputs["input_ids"].to(device)).last_hidden_state
    sampled_latent = model.sample(text_embeddings, latent_size, len(sample_text), guidance_scale=3.0)
    sampled_image = model.decode(sampled_latent)

    for i, image in enumerate(sampled_image):
        image = torch.clip((image + 1) / 2, 0, 1)
        image = image.cpu().permute(1, 2, 0).numpy()
        image = (image * 255).astype(np.uint8)
        image_pil = Image.fromarray(image)
        image_pil.save(os.path.join(save_path, f"test.png"))